In [276]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import preprocessing
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder

import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_predict

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import cross_val_score


In [277]:
df_crop = pd.read_csv('crop_production.csv')
df_crop.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0


In [278]:
df_crop['Crop_Year'].unique()

array([2000, 2001, 2002, 2003, 2004, 2005, 2006, 2010, 1997, 1998, 1999,
       2007, 2008, 2009, 2011, 2012, 2013, 2014, 2015])

In [279]:
df_crop.shape

(246091, 7)

In [280]:
df_crop.isnull().sum()

State_Name          0
District_Name       0
Crop_Year           0
Season              0
Crop                0
Area                0
Production       3730
dtype: int64

In [281]:
df_crop['State_Name'].nunique()

33

In [282]:
df_crop['State_Name'].value_counts()

State_Name
Uttar Pradesh                  33306
Madhya Pradesh                 22943
Karnataka                      21122
Bihar                          18885
Assam                          14628
Odisha                         13575
Tamil Nadu                     13547
Maharashtra                    12628
Rajasthan                      12514
Chhattisgarh                   10709
Andhra Pradesh                  9628
West Bengal                     9613
Gujarat                         8436
Haryana                         5875
Telangana                       5649
Uttarakhand                     4896
Kerala                          4261
Nagaland                        3906
Punjab                          3173
Meghalaya                       2867
Arunachal Pradesh               2546
Himachal Pradesh                2494
Jammu and Kashmir               1634
Tripura                         1412
Manipur                         1267
Jharkhand                       1266
Mizoram                    

In [283]:
df_crop.dropna(subset=['Production'],inplace=True)

In [284]:
df_deleted = df_crop[df_crop['Production'].isnull()]
df_deleted.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production


In [285]:
df_deleted['State_Name'].nunique()

0

In [286]:
df_deleted['State_Name'].value_counts()

Series([], Name: count, dtype: int64)

In [287]:
df_crop.head()

,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0


In [288]:
rain_year = pd.read_csv("rainfall in india 1901-2015.csv")
rain_year.head()

,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,SEP,OCT,NOV,DEC,ANNUAL,Jan-Feb,Mar-May,Jun-Sep,Oct-Dec
0,ANDAMAN & NICOBAR ISLANDS,1901,49.2,87.1,29.2,2.3,528.8,517.5,365.1,481.1,332.6,388.5,558.2,33.6,3373.2,136.3,560.3,1696.3,980.3
1,ANDAMAN & NICOBAR ISLANDS,1902,0.0,159.8,12.2,0.0,446.1,537.1,228.9,753.7,666.2,197.2,359.0,160.5,3520.7,159.8,458.3,2185.9,716.7
2,ANDAMAN & NICOBAR ISLANDS,1903,12.7,144.0,0.0,1.0,235.1,479.9,728.4,326.7,339.0,181.2,284.4,225.0,2957.4,156.7,236.1,1874.0,690.6
3,ANDAMAN & NICOBAR ISLANDS,1904,9.4,14.7,0.0,202.4,304.5,495.1,502.0,160.1,820.4,222.2,308.7,40.1,3079.6,24.1,506.9,1977.6,571.0
4,ANDAMAN & NICOBAR ISLANDS,1905,1.3,0.0,3.3,26.9,279.5,628.7,368.7,330.5,297.0,260.7,25.4,344.7,2566.7,1.3,309.7,1624.9,630.8


In [289]:
df_crop['Season'].unique()

array(['Kharif     ', 'Whole Year ', 'Autumn     ', 'Rabi       ',
       'Summer     ', 'Winter     '], dtype=object)

In [290]:
rain_year['Autumn']=rain_year[['OCT','NOV']].sum(axis=1)/2

In [291]:
rain_year.rename(columns={'Jun-Sep':'Kharif'},inplace=True)
rain_year.rename(columns={'Oct-Dec':'Rabi'},inplace=True)
rain_year.rename(columns={'ANNUAL':'Whole Year'},inplace=True)
rain_year.rename(columns={'Mar-May':'Summer'},inplace=True)


In [292]:
rain_year['Winter']=rain_year[['DEC','Jan-Feb']].sum(axis=1)/2

In [293]:
rain_year.head()

,SUBDIVISION,YEAR,JAN,FEB,MAR,APR,MAY,JUN,JUL,AUG,...,OCT,NOV,DEC,Whole Year,Jan-Feb,Summer,Kharif,Rabi,Autumn,Winter
0,ANDAMAN & NICOBAR ISLANDS,1901,49.2,87.1,29.2,2.3,528.8,517.5,365.1,481.1,...,388.5,558.2,33.6,3373.2,136.3,560.3,1696.3,980.3,473.35,84.95
1,ANDAMAN & NICOBAR ISLANDS,1902,0.0,159.8,12.2,0.0,446.1,537.1,228.9,753.7,...,197.2,359.0,160.5,3520.7,159.8,458.3,2185.9,716.7,278.10,160.15
2,ANDAMAN & NICOBAR ISLANDS,1903,12.7,144.0,0.0,1.0,235.1,479.9,728.4,326.7,...,181.2,284.4,225.0,2957.4,156.7,236.1,1874.0,690.6,232.80,190.85
3,ANDAMAN & NICOBAR ISLANDS,1904,9.4,14.7,0.0,202.4,304.5,495.1,502.0,160.1,...,222.2,308.7,40.1,3079.6,24.1,506.9,1977.6,571.0,265.45,32.10
4,ANDAMAN & NICOBAR ISLANDS,1905,1.3,0.0,3.3,26.9,279.5,628.7,368.7,330.5,...,260.7,25.4,344.7,2566.7,1.3,309.7,1624.9,630.8,143.05,173.00


In [294]:
df= rain_year.drop(columns=['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC'],axis=1)

In [295]:
df.head()

,SUBDIVISION,YEAR,Whole Year,Jan-Feb,Summer,Kharif,Rabi,Autumn,Winter
0,ANDAMAN & NICOBAR ISLANDS,1901,3373.2,136.3,560.3,1696.3,980.3,473.35,84.95
1,ANDAMAN & NICOBAR ISLANDS,1902,3520.7,159.8,458.3,2185.9,716.7,278.10,160.15
2,ANDAMAN & NICOBAR ISLANDS,1903,2957.4,156.7,236.1,1874.0,690.6,232.80,190.85
3,ANDAMAN & NICOBAR ISLANDS,1904,3079.6,24.1,506.9,1977.6,571.0,265.45,32.10
4,ANDAMAN & NICOBAR ISLANDS,1905,2566.7,1.3,309.7,1624.9,630.8,143.05,173.00


In [296]:
df.shape

(4116, 9)

In [297]:
df= df[df['YEAR']>=1997]
df.reset_index(drop=True,inplace=True)
df.head()    
    

,SUBDIVISION,YEAR,Whole Year,Jan-Feb,Summer,Kharif,Rabi,Autumn,Winter
0,ANDAMAN & NICOBAR ISLANDS,1997,2755.1,9.5,296.9,1988.8,459.9,210.75,23.95
1,ANDAMAN & NICOBAR ISLANDS,1998,2846.4,0.9,348.9,1561.2,935.4,423.20,44.95
2,ANDAMAN & NICOBAR ISLANDS,1999,2699.7,91.4,542.2,1358.0,708.1,281.20,118.55
3,ANDAMAN & NICOBAR ISLANDS,2000,2763.2,112.0,812.2,1244.2,594.7,239.75,113.60
4,ANDAMAN & NICOBAR ISLANDS,2001,3080.9,104.7,878.7,1515.4,582.0,245.10,98.25


In [298]:
df.shape

(684, 9)

In [299]:
df_crop.drop('District_Name',axis=1,inplace=True)

In [300]:
df.drop('Jan-Feb',axis=1,inplace=True)

In [301]:
df.rename(columns={'SUBDIVISION':'State_Name'},inplace=True)

In [302]:
df['State_Name'] = df['State_Name'].str.lower()

In [303]:
# Replaces '&' with 'and' in the entire 'State_Name' column
df['State_Name'] = df['State_Name'].str.replace('&', 'and')

In [304]:
df_crop['State_Name'] = df_crop['State_Name'].str.lower()

In [305]:
df.head()

,State_Name,YEAR,Whole Year,Summer,Kharif,Rabi,Autumn,Winter
0,andaman and nicobar islands,1997,2755.1,296.9,1988.8,459.9,210.75,23.95
1,andaman and nicobar islands,1998,2846.4,348.9,1561.2,935.4,423.20,44.95
2,andaman and nicobar islands,1999,2699.7,542.2,1358.0,708.1,281.20,118.55
3,andaman and nicobar islands,2000,2763.2,812.2,1244.2,594.7,239.75,113.60
4,andaman and nicobar islands,2001,3080.9,878.7,1515.4,582.0,245.10,98.25


In [306]:
df_crop.head()

,State_Name,Crop_Year,Season,Crop,Area,Production
0,andaman and nicobar islands,2000,Kharif,Arecanut,1254.0,2000.0
1,andaman and nicobar islands,2000,Kharif,Other Kharif pulses,2.0,1.0
2,andaman and nicobar islands,2000,Kharif,Rice,102.0,321.0
3,andaman and nicobar islands,2000,Whole Year,Banana,176.0,641.0
4,andaman and nicobar islands,2000,Whole Year,Cashewnut,720.0,165.0


In [307]:
# 1. Melt the rainfall dataframe to make it "long" instead of "wide"
# I am assuming your top dataframe is named 'df'
df_rain_melted = df.melt(
    id_vars=['State_Name', 'YEAR'], 
    value_vars=['Whole Year', 'Summer', 'Kharif', 'Rabi', 'Autumn', 'Winter'],
    var_name='Season', 
    value_name='Rainfall'
)

# Check the result - you should now see a 'Season' column and a 'Rainfall' column
print(df_rain_melted.head())

                    State_Name  YEAR      Season  Rainfall
0  andaman and nicobar islands  1997  Whole Year    2755.1
1  andaman and nicobar islands  1998  Whole Year    2846.4
2  andaman and nicobar islands  1999  Whole Year    2699.7
3  andaman and nicobar islands  2000  Whole Year    2763.2
4  andaman and nicobar islands  2001  Whole Year    3080.9


In [308]:
df_rain_melted.shape

(4104, 4)

In [309]:
df_rain_melted[df_rain_melted['Season']=='Kharif']

,State_Name,YEAR,Season,Rainfall
1368,andaman and nicobar islands,1997,Kharif,1988.8
1369,andaman and nicobar islands,1998,Kharif,1561.2
1370,andaman and nicobar islands,1999,Kharif,1358.0
1371,andaman and nicobar islands,2000,Kharif,1244.2
1372,andaman and nicobar islands,2001,Kharif,1515.4
...,...,...,...,...
2047,lakshadweep,2011,Kharif,1013.0
2048,lakshadweep,2012,Kharif,1119.5
2049,lakshadweep,2013,Kharif,1057.0
2050,lakshadweep,2014,Kharif,958.5


In [319]:
# 1. Clean the 'Season' column by removing extra spaces
df_crop['Season'] = df_crop['Season'].str.strip()
df_rain_melted['Season'] = df_rain_melted['Season'].str.strip()

# 2. Clean 'State_Name' again just to be safe
# (Your previous code did this, but let's ensure it's consistent)
df_crop['State_Name'] = df_crop['State_Name'].str.strip().str.lower().str.replace('&', 'and')
df_rain_melted['State_Name'] = df_rain_melted['State_Name'].str.strip().str.lower().str.replace('&', 'and')

# 3. Perform the Merge Again
df_final = pd.merge(
    df_crop, 
    df_rain_melted, 
    left_on=['State_Name', 'Crop_Year', 'Season'], 
    right_on=['State_Name', 'YEAR', 'Season'], 
    how='left'
)

# 4. Cleanup
df_final.drop(columns=['YEAR'], inplace=True)

# 5. Check the result - You should see actual numbers in 'Rainfall' now!
print("Rows with Missing Rainfall:", df_final['Rainfall'].isnull().sum())
df_final.head()

Rows with Missing Rainfall: 174191


,State_Name,Crop_Year,Season,Crop,Area,Production,Rainfall
0,andaman and nicobar islands,2000,Kharif,Arecanut,1254.0,2000.0,1244.2
1,andaman and nicobar islands,2000,Kharif,Other Kharif pulses,2.0,1.0,1244.2
2,andaman and nicobar islands,2000,Kharif,Rice,102.0,321.0,1244.2
3,andaman and nicobar islands,2000,Whole Year,Banana,176.0,641.0,2763.2
4,andaman and nicobar islands,2000,Whole Year,Cashewnut,720.0,165.0,2763.2


In [311]:
df_rain_melted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4104 entries, 0 to 4103
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   State_Name  4104 non-null   object 
 1   YEAR        4104 non-null   int64  
 2   Season      4104 non-null   object 
 3   Rainfall    4099 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 128.4+ KB


In [312]:
df_crop.info()

<class 'pandas.core.frame.DataFrame'>
Index: 242361 entries, 0 to 246090
Data columns (total 6 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   State_Name  242361 non-null  object 
 1   Crop_Year   242361 non-null  int64  
 2   Season      242361 non-null  object 
 3   Crop        242361 non-null  object 
 4   Area        242361 non-null  float64
 5   Production  242361 non-null  float64
dtypes: float64(2), int64(1), object(3)
memory usage: 12.9+ MB


In [316]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242361 entries, 0 to 242360
Data columns (total 7 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   State_Name  242361 non-null  object 
 1   Crop_Year   242361 non-null  int64  
 2   Season      242361 non-null  object 
 3   Crop        242361 non-null  object 
 4   Area        242361 non-null  float64
 5   Production  242361 non-null  float64
 6   Rainfall    0 non-null       float64
dtypes: float64(3), int64(1), object(3)
memory usage: 12.9+ MB
